In [1]:
import pandas as pd
import numpy as np
import datetime
import yfinance as yf

from finrl.meta.preprocessor.yahoodownloader import YahooDownloader
from finrl.meta.preprocessor.preprocessors import FeatureEngineer, data_split
from finrl import config_tickers
from finrl.config import INDICATORS

import itertools

In [2]:
config_tickers.DOW_30_TICKER

['AXP',
 'AMGN',
 'AAPL',
 'BA',
 'CAT',
 'CSCO',
 'CVX',
 'GS',
 'HD',
 'HON',
 'IBM',
 'INTC',
 'JNJ',
 'KO',
 'JPM',
 'MCD',
 'MMM',
 'MRK',
 'MSFT',
 'NKE',
 'PG',
 'TRV',
 'UNH',
 'CRM',
 'VZ',
 'V',
 'WBA',
 'WMT',
 'DIS',
 'DOW']

In [2]:
TRAIN_START_DATE = '2016-01-01'
TRAIN_END_DATE = '2019-01-01'
TRADE_START_DATE = '2019-01-01'
TRADE_END_DATE = '2019-07-01'

In [4]:
df_raw = YahooDownloader(start_date = TRAIN_START_DATE,
                     end_date = TRADE_END_DATE,
                     ticker_list = config_tickers.DOW_30_TICKER).fetch_data()

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%********

Shape of DataFrame:  (25533, 8)


In [5]:
df_raw.head()

Price,date,close,high,low,open,volume,tic,day
0,2016-01-04,23.834375,26.342501,25.500000,25.652500,270597600,AAPL,0
1,2016-01-04,120.479263,159.220001,156.089996,159.000000,5083200,AMGN,0
2,2016-01-04,58.839695,68.180000,66.769997,68.089996,9248300,AXP,0
3,2016-01-04,126.005096,141.699997,139.429993,141.380005,5719500,BA,0
4,2016-01-04,53.563431,68.080002,65.720001,66.879997,8586900,CAT,0


In [6]:
fe = FeatureEngineer(use_technical_indicator=True,
                     tech_indicator_list = INDICATORS,
                     use_vix=True,
                     use_turbulence=True,
                     user_defined_feature = False)

processed = fe.preprocess_data(df_raw)

[*********************100%***********************]  1 of 1 completed

Successfully added technical indicators


Shape of DataFrame:  (877, 8)
Successfully added vix
Successfully added turbulence index


In [7]:
list_ticker = processed["tic"].unique().tolist()
list_date = list(pd.date_range(processed['date'].min(),processed['date'].max()).astype(str))
combination = list(itertools.product(list_date,list_ticker))

processed_full = pd.DataFrame(combination,columns=["date","tic"]).merge(processed,on=["date","tic"],how="left")
processed_full = processed_full[processed_full['date'].isin(processed['date'])]
processed_full = processed_full.sort_values(['date','tic'])

processed_full = processed_full.fillna(0)

In [8]:
processed_full.head()

,date,tic,close,high,low,open,volume,day,macd,boll_ub,boll_lb,rsi_30,cci_30,dx_30,close_30_sma,close_60_sma,vix,turbulence
0,2016-01-04,AAPL,23.834375,26.342501,25.500000,25.652500,270597600.0,0.0,0.0,24.380416,22.691057,0.0,-66.666667,100.0,23.834375,23.834375,20.700001,0.0
1,2016-01-04,AMGN,120.479263,159.220001,156.089996,159.000000,5083200.0,0.0,0.0,24.380416,22.691057,0.0,-66.666667,100.0,120.479263,120.479263,20.700001,0.0
2,2016-01-04,AXP,58.839695,68.180000,66.769997,68.089996,9248300.0,0.0,0.0,24.380416,22.691057,0.0,-66.666667,100.0,58.839695,58.839695,20.700001,0.0
3,2016-01-04,BA,126.005096,141.699997,139.429993,141.380005,5719500.0,0.0,0.0,24.380416,22.691057,0.0,-66.666667,100.0,126.005096,126.005096,20.700001,0.0
4,2016-01-04,CAT,53.563431,68.080002,65.720001,66.879997,8586900.0,0.0,0.0,24.380416,22.691057,0.0,-66.666667,100.0,53.563431,53.563431,20.700001,0.0


In [9]:
train = data_split(processed_full, TRAIN_START_DATE,TRAIN_END_DATE)
trade = data_split(processed_full, TRADE_START_DATE,TRADE_END_DATE)
print(len(train))
print(len(trade))

21866
3567


In [ ]:
train.to_csv('train_data.csv')
trade.to_csv('trade_data.csv')

In [3]:
df_dji = YahooDownloader(start_date = TRADE_START_DATE,
                     end_date = TRADE_END_DATE,
                     ticker_list = ['dji']).fetch_data()

[*********************100%***********************]  1 of 1 completed

Shape of DataFrame:  (104, 8)


In [4]:
df_dji.to_csv('dji30.csv')